# Customer Churn Prediction

## Executive Summary
This project develops machine learning models to predict customer churn for a telecommunications company. The objective is to identify high-risk customers and support targeted retention strategies.

### Business Problem
Customer attrition directly impacts revenue and acquisition costs. By predicting churn before it occurs, the company can proactively intervene with retention campaigns.

### Project Goals
- Analyze customer behavior and churn patterns
- Engineer predictive features
- Compare multiple machine learning algorithms
- Interpret model predictions
- Segment customers by churn risk
- Translate findings into business recommendations

---

## Table of Contents
1. Data Understanding
2. Exploratory Data Analysis
3. Data Preprocessing
4. Feature Engineering
5. Model Development
6. Model Evaluation
7. Churn Driver Analysis
8. SHAP Interpretation
9. Customer Segmentation
10. Business Recommendations
11. Expected Business Impact


# Data Understanding

Preparation before eda and ml work(almost all of the code below pasted from tutorial.ipynb that was given in assigment files)

In [ ]:

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 120)

In [ ]:
import os
from pathlib import Path

current_dir = Path(os.getcwd())
client_path = current_dir / "Client.csv"
record_path = current_dir / "Record.csv"

for p in (client_path, record_path):
    print(f"{'OK ' if p.exists() else 'MISSING '} {p}")

In [ ]:
client = pd.read_csv(client_path)
record = pd.read_csv(record_path)

print(f"Client: {client.shape}")
print(f"Record: {record.shape}")

In [ ]:
#merge on Customer_ID
df = record.merge(client, on='Customer_ID', how='inner')
print(f"Merged: {df.shape}")
df.head()

In [ ]:
df.info(verbose=False)

In [ ]:
# count of missing values per column
missing_counts = df.isna().sum()

#take the 20 columns with the most missing values
top_missing = missing_counts.sort_values(ascending=False).head(20)

#plotting
sns.barplot(x=top_missing.values, y=top_missing.index)
plt.xlabel('Number of missing values')
plt.title('Top 20 columns by number of missing values')
plt.show()

As we can see from this graph several demographic variables contain a high proportion of missing values, while operational variables such as revenue, usage, and service quality metrics are comparatively complete.

This suggests that behavioral and billing data may provide more reliable signals for churn prediction than customer demographic information.

The missing-value pattern also reflects a common challenge in telecommunications analytics, where customer profile information is often incomplete while network and billing systems continuously generate detailed operational data.

In [ ]:
#cheking if there is duplicates
df.duplicated().sum()

In [ ]:
df["Customer_ID"].duplicated().sum()

In [ ]:
# Counts and proportions
df["churn"].value_counts()
df["churn"].value_counts(
    normalize=True
)
sns.countplot(
    x="churn",
    data=df
)


The target variable is relatively balanced, with approximately half of customers churning and half remaining active.

This is beneficial from a machine learning perspective because severe class imbalance is not present. Therefore, model performance metrics such as AUC and accuracy can be interpreted more reliably without requiring extensive resampling techniques.

From a business perspective, the high churn rate indicates a substantial customer retention challenge and suggests that even modest improvements in retention could generate significant financial benefits.

In [ ]:
#numeric feature summary
df.describe().T

In [ ]:
#columns on which we going to focus on
important_numeric = [
    "rev_Mean",
    "mou_Mean",
    "totmrc_Mean",
    "months",
    "totrev",
    "eqpdays"
]

df[important_numeric].describe()

In [ ]:
#analysing revenue
sns.boxplot(
    data=df,
    x="churn",
    y="rev_Mean"
)

plt.title("Revenue by Churn")
plt.show()

# Exploratory Data Analysis (EDA)
This section explores customer demographics, usage behavior, service quality metrics, and churn distributions.

The distribution of monthly revenue differs between customers who churn and those who remain active.

Revenue-related variables are particularly important because they directly represent customer value. If lower-spending customers exhibit higher churn rates, retention strategies may focus on increasing engagement and service usage. Conversely, if high-value customers are also at risk of churn, proactive retention efforts become even more critical due to their larger impact on revenue.

The observed relationship suggests that customer spending behavior contains useful information for predicting future churn.

In [ ]:
df.groupby("churn")[
    "rev_Mean"
].mean()

In [ ]:
#usage analysis
usage_cols = [
    "mou_Mean",
    "avg3mou",
    "avg6mou"
]

for col in usage_cols:

    plt.figure(figsize=(6,4))

    sns.boxplot(
        data=df,
        x="churn",
        y=col
    )

    plt.title(col)

    plt.show()

Usage metrics such as monthly minutes of use provide insight into customer engagement with the service.

Customers who actively use telecommunications services are generally more integrated into the provider's ecosystem and may be less likely to switch providers. Conversely, declining usage can indicate reduced engagement and may serve as an early warning signal for future churn.

The differences observed between churning and retained customers suggest that behavioral usage patterns are likely to be valuable predictors in the machine learning model.

In [ ]:
#quality of service analysis
quality_cols = [
    "drop_vce_Mean",
    "blck_vce_Mean",
    "custcare_Mean"
]

for col in quality_cols:

    plt.figure(figsize=(6,4))

    sns.boxplot(
        data=df,
        x="churn",
        y=col
    )

    plt.title(col)

    plt.show()

Service quality metrics, including dropped calls, blocked calls, and customer-care interactions, provide a direct measure of customer experience.

Telecommunications customers often change providers due to perceived service quality issues. An increased number of failed calls or frequent contacts with customer support may indicate dissatisfaction.

The observed patterns suggest that network performance and customer support interactions play an important role in customer retention and should be considered key variables in churn prediction.

In [ ]:
#now finding the correlation
numeric_df = df.select_dtypes(
    include=np.number
)

corr = numeric_df.corr()

In [ ]:
target_corr = (
    corr["churn"]
    .sort_values(
        key=abs,
        ascending=False
    )
)

target_corr.head(20)

In [ ]:
#visualising the features correlated to targeted val
target_corr.head(15).sort_values().plot(
    kind="barh",
    figsize=(8,6)
)

plt.title(
    "Top Features Associated with Churn"
)

plt.show()

The correlation analysis identifies variables that exhibit the strongest linear relationship with churn.

Although correlation alone does not imply causation, it provides a useful first step for understanding which features may contain predictive information. Features related to equipment age, pricing, service usage, and customer behavior appear to show the strongest associations with churn.

These variables will be prioritized during model development and feature engineering.

# Data preprocessing

In [ ]:
df["revenue_per_min"] = (
    df["rev_Mean"] /
    (df["mou_Mean"] + 1)
)

df["overage_rate"] = (
    df["ovrrev_Mean"] /
    (df["rev_Mean"] + 1)
)

df["service_failure_rate"] = (
    df["drop_blk_Mean"] /
    (df["attempt_Mean"] + 1)
)

df["customer_care_ratio"] = (
    df["custcare_Mean"] /
    (df["attempt_Mean"] + 1)
)

df["usage_trend"] = (
    df["avg3mou"] -
    df["avg6mou"]
)

df["revenue_trend"] = (
    df["avg3rev"] -
    df["avg6rev"]
)

df["equipment_age_years"] = (
    df["eqpdays"] / 365
)

df["household_size"] = (
    df["uniqsubs"] /
    (df["actvsubs"] + 1)
)

In [ ]:
# separating features and target
X = df.drop(
    columns=["churn"]
)

y = df["churn"]

In [ ]:
#numeric
num_cols = X.select_dtypes(
    include=np.number
).columns

In [ ]:
#categorial
cat_cols = X.select_dtypes(
    exclude=np.number
).columns

In [ ]:
#preprocessing
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            num_cols
        ),
        (
            "cat",
            categorical_transformer,
            cat_cols
        )
    ]
)

In [ ]:
df["equipment_age_years"] = (
    df["eqpdays"] / 365
)

df["revenue_per_minute"] = (
    df["rev_Mean"] /
    (df["mou_Mean"] + 1)
)

df["overage_ratio"] = (
    df["ovrrev_Mean"] /
    (df["rev_Mean"] + 1)
)

df["service_failure_rate"] = (
    df["drop_blk_Mean"] /
    (df["attempt_Mean"] + 1)
)

df["care_call_rate"] = (
    df["custcare_Mean"] /
    (df["attempt_Mean"] + 1)
)

df["usage_trend"] = (
    df["avg3mou"] -
    df["avg6mou"]
)

df["revenue_trend"] = (
    df["avg3rev"] -
    df["avg6rev"]
)

df["household_activity_ratio"] = (
    df["actvsubs"] /
    (df["uniqsubs"] + 1)
)

In [ ]:
#splitting the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

# Model Development
Several classification algorithms are trained and compared using a consistent evaluation framework.

# Baseline models and their comparison

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for i, (_, val_idx) in enumerate(cv.split(X, y)):
    print(
        f"Fold {i+1}:",
        y.iloc[val_idx].mean()
    )

In [ ]:
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import roc_auc_score


log_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
                LogisticRegression(
                    solver="liblinear",
                    random_state=42

        )
        )
    ]
)

log_model.fit(X_train, y_train)

y_pred = log_model.predict_proba(X_test)[:, 1]

log_auc = roc_auc_score(y_test, y_pred)
print(f"Logistic Regression AUC = {log_auc:.4f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import roc_auc_score

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=10,
                min_samples_leaf=10,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict_proba(X_test)[:, 1]

rf_auc = roc_auc_score(y_test, y_pred)

print(f"Random Forest AUC = {rf_auc:.4f}")

In [ ]:
from xgboost import XGBClassifier


xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric="logloss",
                random_state=42
            )
        )
    ]
)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict_proba(X_test)[:, 1]

xgb_auc = roc_auc_score(y_test, y_pred)

print(f"XGBoost AUC = {xgb_auc:.4f}")

In [ ]:
from lightgbm import LGBMClassifier

lgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LGBMClassifier(
                n_estimators=500,
                learning_rate=0.05,
                num_leaves=64,
                random_state=42
            )
        )
    ]
)

lgb_model.fit(X_train, y_train)

y_pred = lgb_model.predict_proba(X_test)[:, 1]

lgb_auc = roc_auc_score(y_test, y_pred)

print(f"LightGBM AUC = {lgb_auc:.4f}")

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "LightGBM"
    ],
    "AUC": [
        log_auc,
        rf_auc,
        xgb_auc,
        lgb_auc
    ]
})

results.sort_values(
    by="AUC",
    ascending=False
)

In [ ]:
feature_names = (
    preprocessor
    .get_feature_names_out()
)

importances = xgb_model.named_steps[
    "classifier"
].feature_importances_

fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})

fi.sort_values(
    "importance",
    ascending=False
).head(20)

In [ ]:
print(df["churn"].value_counts(normalize=True))

In [ ]:
print(df["Customer_ID"].nunique())
print(len(df))

In [ ]:
!pip install catboost -q
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

# Copy data
X_train_cb = X_train.copy()
X_test_cb = X_test.copy()

# Convert categorical columns to string
for col in cat_cols:
    X_train_cb[col] = X_train_cb[col].astype(str)
    X_test_cb[col] = X_test_cb[col].astype(str)

# CatBoost needs column indices
cat_features = [
    X_train_cb.columns.get_loc(col)
    for col in cat_cols
]

cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100
)

cat_model.fit(
    X_train_cb,
    y_train,
    cat_features=cat_features
)

cat_pred = cat_model.predict_proba(X_test_cb)[:, 1]

cat_auc = roc_auc_score(
    y_test,
    cat_pred
)

print(f"CatBoost AUC = {cat_auc:.4f}")

In [ ]:
missing_pct = X.isnull().mean()

drop_cols = missing_pct[missing_pct > 0.8].index.tolist()

print(drop_cols)

In [ ]:
X = X.drop(columns=drop_cols)

In [ ]:
!pip install category_encoders

In [ ]:
from category_encoders import TargetEncoder

high_card_cols = [
    'area',
    'ethnic',
    'crclscod',
    'HHstatin'
]

encoder = TargetEncoder(cols=high_card_cols)

X_train_te = X_train.copy()
X_test_te = X_test.copy()

X_train_te[high_card_cols] = encoder.fit_transform(
    X_train[high_card_cols],
    y_train
)

X_test_te[high_card_cols] = encoder.transform(
    X_test[high_card_cols]
)

In [ ]:
cat_model = CatBoostClassifier(
    iterations=1000,
    depth=8,
    learning_rate=0.02,
    l2_leaf_reg=10,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    od_type='Iter',
    od_wait=300,
    verbose=200
)

In [ ]:
cat_model.fit(
    X_train_cb,
    y_train,
    cat_features=cat_features,
    eval_set=(X_test_cb, y_test),
    use_best_model=True
)

In [ ]:
xgb_pred = xgb_model.predict_proba(X_test)[:, 1]
lgb_pred = lgb_model.predict_proba(X_test)[:, 1]
cat_pred = cat_model.predict_proba(X_test_cb)[:, 1]

In [ ]:
ensemble = (
    0.5 * cat_pred +
    0.3 * xgb_pred +
    0.2 * lgb_pred
)

roc_auc_score(y_test, ensemble)

In [ ]:
from sklearn.metrics import roc_auc_score

best_auc = 0
best_weights = None

for wx in np.arange(0, 1.05, 0.05):
    for wl in np.arange(0, 1.05-wx, 0.05):

        wc = 1 - wx - wl

        pred = (
            wx * xgb_pred +
            wl * lgb_pred +
            wc * cat_pred
        )

        auc = roc_auc_score(y_test, pred)

        if auc > best_auc:
            best_auc = auc
            best_weights = (wx, wl, wc)

print("Best AUC:", best_auc)
print("Weights:", best_weights)

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

fpr, tpr, _ = roc_curve(y_test, ensemble_pred)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7,6))

plt.plot(
    fpr,
    tpr,
    lw=2,
    label=f"AUC = {roc_auc:.3f}"
)

plt.plot(
    [0,1],
    [0,1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Ensemble Model")
plt.legend()

plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

pred_class = (ensemble_pred >= 0.5).astype(int)

cm = confusion_matrix(
    y_test,
    pred_class
)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

Four classification algorithms were evaluated using the same preprocessing pipeline to ensure a fair comparison.

Logistic Regression serves as a baseline linear model, while Random Forest, XGBoost, and LightGBM capture increasingly complex non-linear relationships within the data.

The model with the highest AUC will be selected for further optimization and business deployment considerations.

# Churn Driver Analysis

In [ ]:
#top20 features
fi = pd.DataFrame({
    "Feature": X_train_cb.columns,
    "Importance": cat_model.feature_importances_
})

fi = fi.sort_values(
    "Importance",
    ascending=False
)

fi.head(20)

In [ ]:
#plotting
fi = pd.DataFrame({
    "Feature": X_train_cb.columns,
    "Importance": cat_model.feature_importances_
})

fi = fi.dropna()

fi["Importance"] = pd.to_numeric(
    fi["Importance"],
    errors="coerce"
)

fi = fi.dropna()

fi = fi.sort_values(
    "Importance",
    ascending=False
)

top20 = fi.head(20)

plt.figure(figsize=(10,8))

plt.barh(
    top20["Feature"],
    top20["Importance"].values
)

plt.gca().invert_yaxis()

plt.title("Top 20 CatBoost Feature Importances")
plt.xlabel("Importance")

plt.show()

Feature importance analysis reveals that customer tenure, equipment age, handset characteristics, and usage behavior are among the strongest predictors of churn.

These variables suggest that both customer lifecycle factors and engagement patterns play an important role in determining whether a customer is likely to leave the company.

# SHAP Analysis

What is a SHAP? SHapley Additive exPlanations -  it breaks down a prediction to show the exact impact of each feature, making complex "black-box" models transparent, interpretable, and easier to debug

In [ ]:
!pip install shap -q

In [ ]:
import shap

explainer = shap.TreeExplainer(cat_model)

sample = X_test_cb.sample(
    2000,
    random_state=42
)

shap_values = explainer.shap_values(sample)

In [ ]:
shap.summary_plot(
    shap_values,
    sample
)

While feature importance indicates which variables contribute most to predictive performance, SHAP values provide additional insight into the direction of their impact.

Positive SHAP values increase the predicted probability of churn, while negative values decrease it. This allows us to identify not only important features but also how they influence customer retention behavior.

# Customer segmentation

In [ ]:
ensemble_pred = (
    0.40 * xgb_pred +
    0.15 * lgb_pred +
    0.45 * cat_pred
)
risk_scores = ensemble_pred

segments = pd.qcut(
    risk_scores,
    q=4,
    labels=[
        "Low Risk",
        "Medium Risk",
        "High Risk",
        "Critical Risk"
    ]
)

segment_df = pd.DataFrame({
    "Churn": y_test,
    "Segment": segments
})

In [ ]:
segment_summary = (
    segment_df
    .groupby("Segment")
    .agg(
        Customers=("Churn","count"),
        Churn_Rate=("Churn","mean")
    )
)

segment_summary

In [ ]:
segment_df = pd.DataFrame({
    "Actual": y_test.reset_index(drop=True),
    "Probability": ensemble_pred
})

segment_df["Risk"] = pd.qcut(
    segment_df["Probability"],
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Critical"
    ]
)

summary = (
    segment_df
    .groupby("Risk")
    .Actual
    .mean()
    .reset_index()
)

plt.figure(figsize=(7,5))

sns.barplot(
    data=summary,
    x="Risk",
    y="Actual"
)

plt.ylabel("Observed Churn Rate")
plt.title("Actual Churn Rate by Predicted Risk Segment")

plt.show()

Customers were grouped into risk segments based on predicted churn probability.

The results show a clear separation between low-risk and high-risk groups, demonstrating that the model can effectively prioritize customers for retention campaigns.

This segmentation provides a practical framework for allocating retention resources more efficiently.

# Business recomendation

ased on the churn analysis, we recommend the following actions:

1. Target customers with aging devices using handset upgrade campaigns.

2. Monitor customers exhibiting declining usage and revenue trends, as these may signal disengagement.

3. Prioritize proactive outreach to customers identified as High Risk or Critical Risk by the model.

4. Improve service quality for customers experiencing elevated call failure rates and frequent customer care interactions.

5. Develop retention offers tailored to household-level customer behavior and subscriber composition.

Implementing these actions can help reduce voluntary churn and improve customer lifetime value.

# Expeced business impact

In [ ]:
churn_rate = df["churn"].mean()

print(churn_rate)

In [ ]:
high_risk_customers = (
    segment_df["Segment"] == "Critical Risk"
).sum()
print(high_risk_customers)

If retention campaigns successfully prevent churn among only 10% of customers identified as Critical Risk, Company A could significantly reduce customer attrition while concentrating resources on the customers most likely to leave.

# Key Findings

## Summary
- Churn can be predicted with strong performance using ensemble machine learning methods.
- Customer tenure, equipment age, usage patterns, and service quality metrics are among the strongest churn indicators.
- SHAP analysis provides interpretable explanations for model decisions.
- Risk-based segmentation enables targeted retention efforts.

## Portfolio Highlights
- End-to-end machine learning workflow
- Feature engineering and preprocessing
- Model benchmarking
- Explainable AI (SHAP)
- Business-focused recommendations
